In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import threading

# Global variables
drawing = False
points = []
label_options = os.listdir('/home/natsu/MechAI/Dataset')
img = None
img_copy = None
selected_label = None
processing_threads = []
close_all = False  # Global flag to close all windows

def draw_freehand(event, x, y, flags, param):
    global drawing, points, img, img_copy

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        points = [(x, y)]
        img_copy = img.copy()

    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            img_copy = img.copy()
            points.append((x, y))
            for i in range(1, len(points)):
                cv2.line(img_copy, points[i - 1], points[i], (0, 255, 0), 2)
            display_label_options(img_copy)
            cv2.imshow("Image", img_copy)

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        points.append((x, y))
        cv2.line(img_copy, points[0], points[-1], (0, 255, 0), 2)
        display_label_options(img_copy)
        cv2.imshow("Image", img_copy)

def display_label_options(image):
    global selected_label
    height, width = image.shape[:2]

    label_area_width = 300
    font = cv2.FONT_HERSHEY_SIMPLEX
    y_offset = 30

    for i, label in enumerate(label_options):
        color = (255, 255, 255)
        if selected_label == label:
            color = (0, 255, 0)
        cv2.putText(image, f"{i+1}: {label}", (width - label_area_width + 10, y_offset), font, 1, color, 2, cv2.LINE_AA)
        y_offset += 40

def save_cropped_image(img, save_dir):
    global points, selected_label
    if selected_label is None:
        print("No label selected, skipping this image.")
        return

    label_dir = os.path.join(save_dir, selected_label)
    os.makedirs(label_dir, exist_ok=True)
    mask = np.full(img.shape[:2], cv2.GC_PR_BGD, dtype=np.uint8)
    if len(points) > 2:
        points_array = np.array(points, dtype=np.int32)
        cv2.fillPoly(mask, [points_array], cv2.GC_PR_FGD)

        bgd_model = np.zeros((1, 65), np.float64)
        fgd_model = np.zeros((1, 65), np.float64)
        rect = cv2.boundingRect(points_array)
        cv2.grabCut(img, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_MASK)

        mask2 = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 1, 0).astype('uint8')
        result = img * mask2[:, :, np.newaxis]

        x, y, w, h = cv2.boundingRect(points_array)
        cropped_img = result[y:y+h, x:x+w]

        existing_files = glob.glob(os.path.join(label_dir, f"{selected_label}_*.png"))
        file_number = len(existing_files) + 1
        save_path = os.path.join(label_dir, f"{selected_label}_{file_number}.png")
        cv2.imwrite(save_path, cropped_img)
        print(f"Cropped image saved as: {save_path}")
    else:
        print("Shape not drawn. Please draw a complete shape next time.")

def crop_and_label_object(img_input, save_dir):
    global img, img_copy, selected_label, points, close_all
    img = img_input
    img_copy = img.copy()
    selected_label = None
    points.clear()

    cv2.imshow("Image", img_copy)
    cv2.setMouseCallback("Image", draw_freehand)
    display_label_options(img_copy)

    while True:
        key = cv2.waitKey(1) & 0xFF
        if ord('1') <= key <= ord(str(len(label_options))):
            selected_label = label_options[key - ord('1')]
            print(f"Label '{selected_label}' selected.")
            break
        if key == ord('q'):
            print("Quitting current image...")
            break
        if key == 27:  # ESC key
            print("Closing all windows...")
            close_all = True
            break

    cv2.destroyAllWindows()
    if close_all:
        exit(0)  # Exit the program if close_all is set to True

def process_image_in_thread(img, save_dir):
    save_cropped_image(img, save_dir)

def process_images_in_folders(image_folders, save_dir):
    global processing_threads, close_all
    for folder in image_folders:
        image_files = glob.glob(os.path.join(folder, "*.jpg")) + \
                      glob.glob(os.path.join(folder, "*.png")) + \
                      glob.glob(os.path.join(folder, "*.jpeg"))

        for image_path in image_files:
            if close_all:
                print("Processing stopped by user.")
                return

            print(f"Processing image: {image_path}")
            img = cv2.imread(image_path)
            if img is not None:
                crop_and_label_object(img, save_dir)
                if close_all:
                    return
                thread = threading.Thread(target=process_image_in_thread, args=(img.copy(), save_dir))
                thread.start()
                processing_threads.append(thread)
            else:
                print(f"Failed to load image: {image_path}")

    for thread in processing_threads:
        thread.join()

    print("All images processed.")

# Example usage
input_folders = [
    '/home/natsu/MechAI/Dataset/disk_rotor',
    '/home/natsu/MechAI/Dataset/brake_pads'
]
process_images_in_folders(input_folders, '/home/natsu/MechAI/output')


Processing image: /home/natsu/MechAI/Dataset/disk_rotor/disk_rotor_132.png


QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread (0x3323000).
Cannot move to target thread (0x2ef7ab0)

QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread (0x3323000).
Cannot move to target thread (0x2ef7ab0)

QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread (0x3323000).
Cannot move to target thread (0x2ef7ab0)

QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread (0x3323000).
Cannot move to target thread (0x2ef7ab0)

QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread (0x3323000).
Cannot move to target thread (0x2ef7ab0)

QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread (0x3323000).
Cannot move to target thread (0x2ef7ab0)

QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread (0x3323000).
Cannot move to target thread (0x2ef7ab0)

QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread

Label 'disk_rotor' selected.
Processing image: /home/natsu/MechAI/Dataset/disk_rotor/disk_rotor_47.png


QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread (0x3323000).
Cannot move to target thread (0x2ef7ab0)

QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread (0x3323000).
Cannot move to target thread (0x2ef7ab0)

QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread (0x3323000).
Cannot move to target thread (0x2ef7ab0)

QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread (0x3323000).
Cannot move to target thread (0x2ef7ab0)

QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread (0x3323000).
Cannot move to target thread (0x2ef7ab0)

QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread (0x3323000).
Cannot move to target thread (0x2ef7ab0)

QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread (0x3323000).
Cannot move to target thread (0x2ef7ab0)

QObject::moveToThread: Current thread (0x2ef7ab0) is not the object's thread

Cropped image saved as: /home/natsu/MechAI/output/disk_rotor/None_91.png
